In [ ]:
import pandas as pd
import numpy as np
import os
import duckdb

In [ ]:
# Definindo a estrutura de caminhos relativos do projeto
BRONZE_DIR = "../data/bronze"
SILVER_DIR = "../data/silver"

print(f"Lendo de: {BRONZE_DIR}")
print(f"Salvando em: {SILVER_DIR}")

In [ ]:
def ler_nomes_arquivos_bronze():
    """
    Lê os nomes dos arquivos na pasta bronze e retorna uma lista de nomes de arquivos.
    """
    return [f for f in os.listdir(BRONZE_DIR) if os.path.isfile(os.path.join(BRONZE_DIR, f))]

In [ ]:
lista_arquivos = ler_nomes_arquivos_bronze()

In [ ]:
def transformar_csv_para_parquet(lista_arquivos):
    """
    Função para transformar um arquivo CSV em Parquet.
    
    Parâmetros:
    lista_arquivos (list): Lista de nomes dos arquivos CSV de entrada.
    """

    for arquivo in lista_arquivos:
        # Lendo o arquivo CSV
        df = pd.read_csv(os.path.join(BRONZE_DIR, arquivo))
        
        # Definindo o nome do arquivo Parquet de saída
        nome_arquivo_parquet = os.path.splitext(arquivo)[0] + '.parquet'
        
        # Salvando o DataFrame como Parquet
        df.to_parquet(os.path.join(SILVER_DIR, nome_arquivo_parquet), index=False)
        
        print(f"Arquivo {arquivo} transformado")


In [ ]:
transformar_csv_para_parquet(lista_arquivos)

In [ ]:
def ler_nomes_arquivos_silver():
    """
    Lê os nomes dos arquivos na pasta silver e retorna uma lista de nomes de arquivos.
    """
    return [f for f in os.listdir(SILVER_DIR) if os.path.isfile(os.path.join(SILVER_DIR, f))]

In [ ]:
ler_nomes_arquivos_silver()

In [ ]:
operacoes = pd.read_parquet(os.path.join(SILVER_DIR, 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.parquet'))

operacoes.info()

#### Tratamento de Valores Nulls

In [ ]:
def rankear_nullos(df):
    """Rankear colunas por valores nulos e retornar análise se houver nulos.

    Parâmetros:
        df (DataFrame): DataFrame a ser verificado.

    Retorna:
        False | DataFrame: False se não houver nulos; DataFrame com total e porcentagem caso contrário.
    """
    total_nulls = df.isnull().sum()
    total_nulls = total_nulls[total_nulls > 0].sort_values(ascending=False)

    if total_nulls.empty:
        return False

    total_nulls_percentage = ((total_nulls / df.shape[0]) * 100).round(2)
    return pd.DataFrame({'Total Nulls': total_nulls, '%': total_nulls_percentage})


total_nulls_df = rankear_nullos(operacoes)


In [ ]:
total_nulls_df

In [ ]:
operacoes["tipo_excepcionalidade"].value_counts(dropna=False)

**Regra de negócio:** Se _tipo_excepcionalidade_
 está nulo, significa que a operação seguiu o fluxo padrão (sem exceção). Então, podemos realizar a classificação 0 = não, e 1 = sim.

In [ ]:
operacoes["tem_excepcionalidade"] = operacoes["tipo_excepcionalidade"].notna().astype(int)
operacoes[["tipo_excepcionalidade", "tem_excepcionalidade"]].drop_duplicates()

In [ ]:
operacoes = operacoes.drop(columns=["tipo_excepcionalidade"])

In [ ]:
operacoes.isnull().sum().sort_values(ascending=False).head(10)

**Regra de Négocio:** Se o CNPJ da instituição financeira credenciada estiver nulo, podemos entender que a operação foi realizada diretamente com o BNDES, sem intermediação de uma instituição financeira. Portanto, vamos preencher esses valores nulos com a string "OPERAÇÃO DIRETA".

In [ ]:
operacoes["cnpj_instituicao_financeira_credenciada"] = operacoes["cnpj_instituicao_financeira_credenciada"].fillna("0000000000000.0")
operacoes["nome_instituicao_financeira_credenciada"] = operacoes["nome_instituicao_financeira_credenciada"].fillna("OPERAÇÃO DIRETA")
print(f"{operacoes[['cnpj_instituicao_financeira_credenciada']].dtypes}")

In [ ]:
operacoes.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
print(f"{operacoes[['cnpj_cliente']].dtypes}")

In [ ]:
#preenchendo os valores nulos
operacoes["id_municipio"] = operacoes["id_municipio"].fillna("NÃO INFORMADO")
operacoes["cnpj_cliente"] = operacoes["cnpj_cliente"].fillna("00000000000.0")
operacoes["situacao_contrato"] = operacoes["situacao_contrato"].fillna("OUTROS")
operacoes["tipo_fonte_recursos"] = operacoes["tipo_fonte_recursos"].fillna("OUTROS")

In [ ]:
operacoes.head(5)

In [ ]:
#removendo colunas cnae desnecessárias
operacoes = operacoes.drop(columns=["classe_cnae","subclasse_cnae","grupo_cnae","divisao_cnae","secao_cnae"])

In [95]:
def verificar_valores_nulos(df):
    """Verifica se um DataFrame contém valores nulos.

    Parâmetros:
        df (DataFrame): DataFrame a ser verificado.

    Retorna:
        bool: True se houver pelo menos um valor nulo, False caso contrário.
    """
    return df.isnull().any().any()


verificar_valores_nulos(operacoes)


np.False_